# ForensicMacroWood-CITES Quick-Start Tutorial
### A Macroscopic Cross-Sectional Timber Image Benchmark for CITES-Regulated Hardwood Identification

**Elsevier Data in Brief (DiB)** | **Zenodo Scientific Archive DOI:** [10.5281/zenodo.14892180](https://doi.org/10.5281/zenodo.14892180) | **License:** CC BY 4.0

---

## Notebook Overview & Reproducibility Guide
This interactive tutorial provides an end-to-end, reproducible walkthrough for the **ForensicMacroWood-CITES** benchmark:
1. **Metadata & Quality Control**: Inspect the 19 tropical hardwood taxa in family Fabaceae, CITES Appendix II conservation status, physical specimen cohort counts, focus sharpness, and bitwise SHA-256 cryptographic checksums.
2. **Governed Partition Audit**: Verify complete Class Coverage Rate (CCR = 100.0%) and zero bitwise cross-split duplicate captures (SHA-256 Overlap = 0).
3. **Supervised Classification Baseline**: Evaluate ConvNeXt-Tiny baseline performance under Multiclass Focal Loss.
4. **Taxon Performance Visualization**: Inspect per-species F1-score distributions.
5. **Simulated Customs Inspection Query**: Demonstrate automated forensic timber verification for border-control enforcement.

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["font.size"] = 10
plt.rcParams["figure.dpi"] = 110
print("Libraries loaded successfully.")

In [ ]:
# Step 1: Load and Inspect Master Metadata
meta_path = Path("../metadata/metadata.csv")
if not meta_path.exists():
    meta_path = Path("metadata/metadata.csv")
df = pd.read_csv(meta_path)
print(f"Total Curated Images : {len(df)}")
print(f"Botanical Genera     : {df['genus'].nunique()} genera in family Fabaceae")
print(f"Total Species (Spp.) : {df['class_name'].nunique()} species")
df[['image_id', 'genus', 'species', 'cites_status', 'laplacian_var']].head()

In [ ]:
# Step 2: Governed Partition Integrity
train_n = (df['split'] == 'train').sum()
val_n = (df['split'] == 'val').sum()
test_n = (df['split'] == 'test').sum()
print(f"Split Allocation: Train={train_n} | Val={val_n} | Test={test_n}")
train_classes = set(df[df['split'] == 'train']['class_name'])
test_classes = set(df[df['split'] == 'test']['class_name'])
print(f"Class Coverage Rate: Train CCR = {len(train_classes)/df['class_name'].nunique()*100:.1f}% | Test CCR = {len(test_classes)/df['class_name'].nunique()*100:.1f}%")

In [ ]:
# Step 3: Supervised Classification Baseline Metrics (ConvNeXt-Tiny)
res_path = Path("../classification_output/classification_results_focal.json")
if not res_path.exists():
    res_path = Path("classification_output/classification_results_focal.json")
with open(res_path, 'r', encoding='utf-8') as f:
    results = json.load(f)
sm = results['summary_metrics']
print("ConvNeXt-Tiny Baseline Results:")
print(f"- Top-1 Accuracy : {sm['overall_accuracy']*100:.2f}%")
print(f"- Macro Precision: {sm['macro_precision']*100:.2f}%")
print(f"- Macro Recall   : {sm['macro_recall']*100:.2f}%")
print(f"- Macro F1-Score : {sm['macro_f1']*100:.2f}%")

In [ ]:
# Step 4: Taxon-Level Performance Visualization
pcm = results['per_class_metrics']
taxa = sorted(pcm.keys())
f1_scores = [pcm[t]['f1_score'] * 100 for t in taxa]
plt.figure(figsize=(10, 6))
bars = plt.barh(taxa, f1_scores, color='#1f77b4', edgecolor='black', height=0.65)
plt.axvline(90, color='red', linestyle='--', label='F1 = 90% Threshold')
plt.xlabel('F1-Score (%)')
plt.title('ConvNeXt-Tiny Per-Species Test F1-Score (ForensicMacroWood-CITES)')
plt.xlim(0, 105)
plt.gca().invert_yaxis()
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()